In [1]:
import pandas as pd
from google.cloud import bigquery

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, cross_val_score



client = bigquery.Client(project="weather-pipeline-cevikyi")   # gcloud girişinle bağlanır
df = client.query("""
    select weather_date, temp_max
    from analytics.daily_weather
    where weather_date <= current_date('Europe/Istanbul')   -- bitmemiş günleri alma
    order by weather_date
""").to_dataframe()                                        # sorguyu çalıştır, DataFrame yap
df["weather_date"] = pd.to_datetime(df["weather_date"])    # BigQuery'nin tarih tipini pandas'ınkine çevir


/Users/cevikyi/weather-pipeline/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.10). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/cevikyi/weather-pipeline/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.10). Google will not post any further updates to google.cloud.bigquery supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.cloud.bigquery.
  warnings.warn(message, FutureWarning)
/Users/cevikyi/weather-pipeline/.venv/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2122: UserWarning: BigQuery Storage module not f

In [2]:
df.tail()

,weather_date,temp_max
1784,2026-11-20,21.7
1785,2026-11-21,17.8
1786,2026-11-22,16.4
1787,2026-11-23,20.1
1788,2026-11-24,11.6


In [3]:
df["target"] = df["temp_max"].shift(-1)

df["lag1"]= df["temp_max"].shift(1)
df["lag2"]= df["temp_max"].shift(2)
df["lag7"]= df["temp_max"].shift(7)
df["lag28"]= df["temp_max"].shift(28)

df["temp_rolling_3"] = df["temp_max"].rolling(3).mean()
df["temp_rolling_7"] = df["temp_max"].rolling(7).mean()

df["temp_rolling_28"] = df["temp_max"].rolling(28).mean()

df["month"] = df["weather_date"].dt.month
df["day_of_year"] = df["weather_date"].dt.dayofyear

df.tail()

,weather_date,temp_max,target,lag1,lag2,lag7,lag28,temp_rolling_3,temp_rolling_7,temp_rolling_28,month,day_of_year
1784,2026-11-20,21.7,17.8,20.9,18.7,20.9,22.5,20.433333,18.157143,17.192857,11,324
1785,2026-11-21,17.8,16.4,21.7,20.9,20.8,18.8,20.133333,17.728571,17.157143,11,325
1786,2026-11-22,16.4,20.1,17.8,21.7,13.3,20.5,18.633333,18.171429,17.010714,11,326
1787,2026-11-23,20.1,11.6,16.4,17.8,14.6,21.0,18.100000,18.957143,16.978571,11,327
1788,2026-11-24,11.6,NaN,20.1,16.4,17.1,16.7,16.033333,18.171429,16.796429,11,328


In [4]:
import numpy as np

gun = (df["weather_date"] + pd.Timedelta(days=1)).dt.dayofyear   # tahmin edilen günün yılın kaçıncı günü olduğu
df["sin"] = np.sin(2 * np.pi * gun / 365.25)
df["cos"] = np.cos(2 * np.pi * gun / 365.25)

In [5]:
features = [
    "temp_max",
    "lag1",
    "lag2",
    "lag7",
    "lag28",
    "temp_rolling_3",
    "temp_rolling_7",
    "temp_rolling_28",
    "month",
    "day_of_year",
    "sin", "cos"
]


In [6]:
df = df.sort_values("weather_date").reset_index(drop=True)

model_df = df.dropna(subset=features + ["target"])

# 1. Final test setini ayır
development = model_df.iloc[:-365]
final_test = model_df.iloc[-365:]

X_dev = development[features]
y_dev = development["target"]

X_test = final_test[features]
y_test = final_test["target"]

In [7]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

tscv = TimeSeriesSplit(n_splits=12, test_size=30, gap=1)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42)
}

results = {}

for name, model in models.items():

    scores = -cross_val_score(
        model,
        X_dev,
        y_dev,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    results[name] = scores.mean()

print(results)

{'Linear Regression': np.float64(1.6948503423339016), 'Random Forest': np.float64(1.783766666666667), 'XGBoost': np.float64(1.944672778050105)}


In [8]:
# En düşük MAE'ye sahip modeli seç
best_model_name = min(results, key=results.get)
best_model = models[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"CV MAE: {results[best_model_name]:.3f}")

Best Model: Linear Regression
CV MAE: 1.695


In [ ]:
print("baseline  ", mean_absolute_error(y_test, final_test["temp_max"]))   # yarın = bugün

baseline   1.9421917808219178


AttributeError: 'str' object has no attribute 'fit'

In [ ]:
best_model.fit(X_dev, y_dev)
print(best_model_name, mean_absolute_error(y_test, best_model.predict(X_test)))   # final test

best_model.fit(model_df[features], model_df["target"])                            # bütün veriyle eğit
print("yarın", best_model.predict(df[features].tail(1)))

linear 1.7412739976643479
yarın [11.48495113]
